# 06 · Road following — iteration 3: dynamic throttle

**Author:** Albert Sagré Martínez.

**Changes vs. iteration 2**
- First speed scheduler: throttle is increased on straights (|steering| ≤ 0.3) and reduced in corners (|steering| > 0.3) within a fixed band, ±0.01 per frame.
- Camera ISP override installed for the IMX219 sensor (fixes colour/vignetting, see README).
- Camera at 15 fps.

Code kept as run on the car (archival).

# Road Following + dynamic throttling

In [ ]:
import torch
from torch2trt import TRTModule

model_trt = TRTModule()
model_trt.load_state_dict(torch.load('road_following_model_uni_V3_trt.pth'))

In [ ]:
from jetracer.nvidia_racecar import NvidiaRacecar

car = NvidiaRacecar()

In [ ]:
from jetcam.csi_camera import CSICamera

camera = CSICamera(width=224, height=224, capture_fps=15)

In [ ]:
import traitlets
from IPython.display import display
from ipywidgets import Layout, Button, Box
import ipywidgets.widgets as widgets

network_output_slider = widgets.FloatSlider(description='Network Output', min=-1.0, max=1.0, value=0, step=0.01, orientation='horizontal', disabled=False, layout={'width': '500px'})
steering_gain_slider  = widgets.FloatSlider(description='Steering Gain', min=-1.0, max=1.0, value=-0.65, step=0.01, orientation='horizontal', layout={'width': '400px'})
steering_bias_slider  = widgets.FloatSlider(description='Steering Bias', min=-0.5, max=0.5, value=0.0, step=0.01, orientation='horizontal', layout={'width': '400px'})
steering_value_slider = widgets.FloatSlider(description='Steering', min=-1.0, max=1.0, value=0, step=0.01, orientation='horizontal', disabled=False, layout={'width': '500px'})
throttle_slider = widgets.FloatSlider(description='Throttle', min=0.0, max=0.2, value=0.0, step=0.01, orientation='vertical')
#  throttle_slider value should be [-1,1] set to [0,0.2] for prevention

steering_gain_link   = traitlets.link((steering_gain_slider, 'value'), (car, 'steering_gain'))
steering_offset_link = traitlets.link((steering_bias_slider, 'value'), (car, 'steering_offset'))
#steering_value_link  = traitlets.link((steering_value_slider, 'value'), (car, 'steering'))
throttle_slider_link = traitlets.link((throttle_slider, 'value'), (car, 'throttle'))



In [ ]:
from utils import preprocess
from jetcam.utils import bgr8_to_jpeg

def update(change):
    global blocked_slider, robot
    new_image = change['new']
    
    image = preprocess(new_image).half()
    output = model_trt(image).detach().cpu().numpy().flatten()
    x = float(output[0])
    y = float(output[0])
    
    network_output_slider.value = x
    steering = x * steering_gain_slider.value + steering_bias_slider.value
    
    if(steering<-1.0):
        steering_value_slider.value = -1.0
    elif(steering>1.0):
        steering_value_slider.value = 1.0
    else:
        steering_value_slider.value = steering 
    car.steering = x
    
      
    if (steering<=0.3 and steering>=-0.3 and throttle_slider.value<=0.16 and throttle_slider.value>0.12):
        throttle_slider.value += 0.01
    elif ((steering>0.3 or steering<-0.3) and throttle_slider.value>0.14):
        throttle_slider.value += -0.01      
    else:
        throttle_slider.value = throttle_slider.value
          
    
update({'new': camera.value})

In [ ]:
camera.observe(update, names='value') 
camera.running = True

In [ ]:
display(
    widgets.HBox(
        [widgets.VBox([network_output_slider,
                       widgets.Label(value="X"),
                       steering_gain_slider,
                       widgets.Label(value="+"),
                       steering_bias_slider,
                       widgets.Label(value="||"), 
                       steering_value_slider], layout=Layout(align_items='center')
                     ),
         throttle_slider]
                )
        )